In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model

e:\RAG AGENTIC AI\RAG LEARNING\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [3]:
import os 
from dotenv import load_dotenv
load_dotenv()

python-dotenv could not parse statement starting at line 7


True

In [5]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [6]:
loader = TextLoader("langchain_rag_dataset.txt")
row_docs  = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size =300,chunk_overlap =50)
chunks = splitter.split_documents(row_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [8]:
# Step 2: FAISS Vector Store with Hugging Face Embeddings
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks,embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 281.91it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# step 3 : Create MMR Retriver
retriver = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":3}
)

In [10]:
# Step 4 : Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.
                                      
Context:
{context}

Question : {input}

""")
llm=init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq"
)

In [11]:
# Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriver,combine_docs_chain=document_chain)

In [12]:
# Step 6 : Query
query = {"input": "How does Langchain support agents and memory?"}
response = rag_chain.invoke(query)
print("Answer: \n",response["answer"])

Answer: 
 Based on the provided context, LangChain supports agents and memory in the following ways:

1. **Multi-turn Conversations**: LangChain helps models retain previous interactions, making multi-turn conversations more coherent.

2. **Conversational Memory**: LangChain supports conversational memory using two classes:
   - **ConversationBufferMemory**: This allows agents to retain previous interactions.
   - **ConversationSummaryMemory**: This provides summarization memory, summarizing previous conversations for agents.

3. **Agent Capabilities**: LangChain agents can:
   - Use tools like calculators, search APIs, or custom functions based on the instructions they receive.
   - Interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.

4. **Agent Decision Making**: LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task, enabling more complex and dynamic interactions.


In [13]:
response

{'input': 'How does Langchain support agents and memory?',
 'context': [Document(id='b0c2729b-702d-49fb-9c2e-e84a072014e5', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='c79dda6a-50db-442b-834b-a1a163378af4', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='7bd57692-c33f-4c3b-8b9b-37f28648c49c', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order duri